## Game Representation · Zero-Sum Games · Nash Equilibrium

### Learning outcomes

1. Represent a game with states, players, actions, transitions, terminal tests, and utilities.
2. Classify games by randomness, information, payoff structure, and move order.
3. Evaluate positions when exhaustive search is not practical.
4. Analyze zero-sum and constant-sum games with pure and mixed strategies.
5. Apply Minimax and explain when Alpha–Beta Pruning can discard a branch.
6. Find best responses, dominant strategies, and pure or mixed Nash equilibria.
7. Distinguish Nash equilibrium from Pareto optimality.


# Import some libraries

In [1]:
import pandas as pd
import numpy as np

# 1. Representation of Games

A game may be represented by

$$
G=\langle S,s_0,P,A,Result,Terminal,Utility\rangle.
$$

The representation should make legal actions, state transitions, terminal outcomes, and player preferences explicit.


## Identify the components of a game

> **Problem description.** Two agents play a take-away game. A pile starts with seven stones. On a turn, an agent removes one or two stones. The agent taking the final stone wins. `MAX` moves first.

1. $S$ $\in \{0, 1, 2, 3, 4, 5, 6, 7\}$. This is the number of stones remaining in the pile
2. $s_0$ — $7$.
3. $P(s)$ — Two agents who are playing the game take turns.
4. $A(s)$ — Removes the stone from the pile.
5. $Result(s,a)$ — Remove one or two stones.
6. $Terminal(s)$ — One of the agents takes the final stone or the pile is empty.
7. $Utility(s,MAX)$ — +1 for win and -1 for lose.

## Implement the take-away representation

> **Problem description.** Use a state tuple `(stones_remaining, player_to_move)`. An action is the number of stones removed. Utility is measured from MAX's perspective: `+1` for a MAX win and `-1` for a MAX loss.


In [17]:
INITIAL_TAKE_AWAY_STATE = (7, "MAX")


def legal_actions(state: tuple[int, str]) -> list[int] | None:
    """Return the number of stones that can be legally taken in a given state.

    Parameters
    ----------
    state:
        A tuple of ``(remaining stones, active player)``.
    """

    # Return [1], [1, 2], or [] depending on the number of remaining stones.
    if state[0] == 0:
        return []
    elif state[0] == 1:
        return [1]
    return [1, 2]


def result(state: tuple[int, str], action: int) -> tuple[int, str] | None:
    """Apply a legal action and switch turns to the other player.

    Parameters
    ----------
    state:
        State before the action.
    action:
        The number of stones to take.
    """

    # Validate action, update the number of stones, and switch MAX ↔ MIN.
    if action not in legal_actions(state):
        raise ValueError
    return (state[0] - action, "MAX" if state[1] == "MIN" else "MIN")


def terminal(state: tuple[int, str]) -> bool | None:
    """Indicate whether the game has ended.

    Parameters
    ----------
    state:
        State to check.
    """

    # The game ends when no stones remain.
    if state[0] == 0:
        return True
    return False


def utility_for_max(state: tuple[int, str]) -> int | None:
    """Return the utility at a terminal state from MAX's perspective.

    Parameters
    ----------
    state:
        Terminal state; the active player is the one who cannot take any stones.
    """

    # Raise an error if the state is not terminal, then return +1 or -1.
    if not terminal(state):
        raise ValueError
        return None
    if state[1] == "MAX":
        return -1
    return 1


def play_actions(
    initial_state: tuple[int, str],
    actions: list[int],
) -> tuple[int, str] | None:
    """Sequentially simulate a list of actions from the initial state.

    Parameters
    ----------
    initial_state:
        The state from which to start the simulation.
    actions:
        List of stones taken in order.
    """

    # Loop through actions and call result after each turn.
    for action in actions:
        initial_state = result(initial_state, action)
    return initial_state

In [5]:
display(play_actions(INITIAL_TAKE_AWAY_STATE, [1, 2, 1, 1]))
display(play_actions(INITIAL_TAKE_AWAY_STATE, [1, 1, 1, 1]))
display(play_actions(INITIAL_TAKE_AWAY_STATE, [1, 1, 2, 1, 1]))
display(play_actions(INITIAL_TAKE_AWAY_STATE, [1, 1, 1, 1, 1]))
display(play_actions(INITIAL_TAKE_AWAY_STATE, [1, 1, 1, 1, 1, 1]))

(2, 'MAX')

(3, 'MAX')

(1, 'MIN')

(2, 'MIN')

(1, 'MAX')

### Reflection

The number of remaining stones alone does not specify which player's turn it is to move, because different sequences of moves could lead to the same number of remaining stones with different active players. By explicitly including `player_to_move` in the state representation, we uniquely identify who must act next, thereby removing any ambiguity.

## Classify games

> **Problem description.** Classify each game using the four dimensions from the lecture: randomness, information, payoff structure, and move order.

Use exactly these labels:

- `Deterministic` or `Stochastic`
- `Perfect` or `Imperfect`
- `Zero-sum` or `General-sum`
- `Sequential` or `Simultaneous`

Assume fees are ignored in the hidden-card duel. In the pricing game, each firm chooses without seeing the rival's current choice, and both firms can earn positive profit.


In [6]:
classification_answers = {
    "Chess": ("Deterministic", "Perfect", "Zero-sum", "Sequential"),
    "Backgammon": ("Stochastic", "Perfect", "Zero-sum", "Sequential"),
    "Hidden-card duel": ("Stochastic", "Imperfect", "Zero-sum", "Sequential"),
    "Simultaneous pricing": ("Deterministic", "Imperfect", "General-sum", "Simultaneous"),
}

pd.DataFrame.from_dict(
    classification_answers,
    orient="index",
)

,0,1,2,3
Chess,Deterministic,Perfect,Zero-sum,Sequential
Backgammon,Stochastic,Perfect,Zero-sum,Sequential
Hidden-card duel,Stochastic,Imperfect,Zero-sum,Sequential
Simultaneous pricing,Deterministic,Imperfect,General-sum,Simultaneous


### Information reflection

In an imperfect-information game, a belief state represents the set of all possible actual game states that are consistent with the player's observations so far. An AI agent needs a probability distribution over these states because it cannot be completely certain of the true state, and acting on a single assumed state could be easily exploited by opponents or lead to highly suboptimal moves.

## Evaluate non-terminal positions

> **Problem description.** A chess agent stops searching before a terminal state and estimates

$$
Eval(s)=9Q+5R+3B+3N+P+Position.
$$

Each feature is the agent's material or positional advantage over its opponent. Calculate the score for both positions and identify which position MAX prefers.


In [7]:
position_a = {"Q": 1, "R": 0, "B": -1, "N": 2, "P": 1, "Position": 0.5}
position_b = {"Q": 0, "R": 1, "B": 0, "N": -1, "P": -2, "Position": -0.5}


def chess_evaluation(features: dict[str, float]) -> float | None:
    """Calculate the chess evaluation score linearly from six features.

    Parameters
    ----------
    features:
        Dictionary of Q, R, B, N, P and Position.
    """

    # Implement the exact formula from the problem description.
    return (9 * features.get("Q", 0) + 5 * features.get("R", 0) + 3 * features.get("B", 0)
    + 3 * features.get("N", 0) + features.get("P", 0) + features.get("Position", 0))


# Calculate both scores and assign "A" or "B" to preferred_position.
score_a = chess_evaluation(position_a)
score_b = chess_evaluation(position_b)
preferred_position = "A" if score_a > score_b else "B"

print(f"Score A: {score_a}")
print(f"Score B: {score_b}")
print(f"Preferred position: {preferred_position}")

Score A: 13.5
Score B: -0.5
Preferred position: A


### Evaluation-function reflection

Give one reason why a higher evaluation score can still lead to a poor move. Your answer should mention either search depth, missing features, or approximation error.

**Answer:**
A higher evaluation score can still lead to a poor move due to limited search depth (the horizon effect), where a highly favorable position is evaluated positively, but a disastrous loss or threat lies just one step beyond the search horizon.

# 2. Zero-Sum Games and Minimax

A two-player zero-sum game satisfies

$$U_1(s)+U_2(s)=0$$

at every outcome. A constant-sum game has the same strategic structure after a suitable shift of utilities, while a general-sum game may combine competition and cooperation.


## Distinguish zero-sum, constant-sum, and general-sum

> **Problem description.** Complete a function that compares two payoff matrices. Return `"zero-sum"` if every cell sums to zero, `"constant-sum"` if every cell has the same nonzero sum, and `"general-sum"` otherwise.


In [18]:
zero_a = np.array([[1, -1], [-1, 1]], dtype=float)
zero_b = -zero_a

constant_a = np.array([[10, 0], [4, 7]], dtype=float)
constant_b = np.array([[0, 10], [6, 3]], dtype=float)

general_a = np.array([[3, 0], [5, 1]], dtype=float)
general_b = np.array([[3, 5], [0, 1]], dtype=float)


def classify_payoff_structure(
    payoff_1: np.ndarray,
    payoff_2: np.ndarray,
) -> str | None:
    """Classify a pair of payoff matrices by the total utility sum in all cells.

    Parameters
    ----------
    payoff_1, payoff_2:
        Two payoff matrices of the same shape.
    """

    # Check shape, calculate the sum matrix, and return one of the three labels.
    if np.shape(payoff_1) != np.shape(payoff_2):
        raise ValueError
    sum_matrix = payoff_1 + payoff_2
    if np.all(sum_matrix == 0):
        return "zero-sum"
    elif np.all(np.abs(sum_matrix) == np.abs(sum_matrix[0, 0])):
        return "constant-sum"
    else:
        return "general-sum"


structure_answers = {
    "Game 1": classify_payoff_structure(zero_a, zero_b),
    "Game 2": classify_payoff_structure(constant_a, constant_b),
    "Game 3": classify_payoff_structure(general_a, general_b),
}
structure_answers

{'Game 1': 'zero-sum', 'Game 2': 'constant-sum', 'Game 3': 'general-sum'}

### Payoff-structure reflection

A competitive situation is not automatically zero-sum because players' interests might not be diametrically opposed, allowing for joint losses or gains. In Game 3 (the Prisoner's Dilemma), if both players defect, they both end up with a payoff of 1 (sum = 2), whereas if they both cooperate, they earn 3 each (sum = 6). Since the sum of payoffs varies depending on the outcome, it is a general-sum game, even though they compete.

## Pure and mixed strategies in Rock–Paper–Scissors

> **Problem description.** Rows and columns use the order `Scissors`, `Rock`, `Paper`. The matrix contains Player A's utility. Complete the expected-payoff function, then analyze a uniform strategy and a biased strategy for A.

For mixed strategies $p$ and $q$,

$$E[U_A]=p^T A q.$$


In [19]:
rps_actions = ["Scissors", "Rock", "Paper"]
rps_payoff = np.array(
    [
        [0, -1, 1],
        [1, 0, -1],
        [-1, 1, 0],
    ],
    dtype=float,
)


def expected_payoff(
    payoff: np.ndarray,
    row_strategy: np.ndarray,
    column_strategy: np.ndarray,
) -> float | None:
    """Calculate the expected payoff of the row player for two mixed strategies.

    Parameters
    ----------
    payoff:
        The payoff matrix of the row player.
    row_strategy, column_strategy:
        Non-negative probability vectors that sum to 1.
    """

    # Verify probability distributions and calculate p^T A q.
    if not np.allclose(np.sum(row_strategy), 1):
        raise ValueError
    if not np.allclose(np.sum(column_strategy), 1):
        raise ValueError
    return float(np.dot(np.transpose(row_strategy), np.dot(payoff, column_strategy)))


uniform_a = np.full(3, 1 / 3)
pure_b_strategies = np.eye(3)

# Calculate the payoffs of uniform_a against each pure strategy of B.
uniform_values_against_b = np.array([
    expected_payoff(rps_payoff, uniform_a, pure_b_strategies[:, i]) for i in range(3)
])

biased_a = np.array([0.60, 0.20, 0.20])

# Calculate the three payoffs and choose the index B uses to minimize A's payoff the most.
biased_values_against_b = np.array([
    expected_payoff(rps_payoff, biased_a, pure_b_strategies[:, i]) for i in range(3)
])
best_response_index_for_b = int(np.argmin(biased_values_against_b))

print(f"Uniform values against B: {uniform_values_against_b}")
print(f"Biased values against B: {biased_values_against_b}")
print(f"Best response index for B: {best_response_index_for_b}")

Uniform values against B: [0. 0. 0.]
Biased values against B: [ 0.  -0.4  0.4]
Best response index for B: 1


### Mixed-strategy reflection

Why can B exploit the biased strategy but not the uniform strategy? Because the biased strategy is predictable (playing Scissors 60% of the time), Player B can unilaterally improve their payoff by always playing Rock to exploit this bias. In contrast, against the uniform strategy (playing each action with 1/3 probability), any pure strategy B plays yields an expected payoff of exactly 0, meaning B cannot exploit A.

## Maximin, minimax, and saddle points

> **Problem description.** The row player MAX selects a row and the column player MIN selects a column. Compute row minima, the maximin value, column maxima, the minimax value, and all pure saddle-point coordinates.

A pure saddle point exists when maximin equals minimax.


In [20]:
game_without_saddle = np.array([[2, -1, 3], [0, 1, -2]], dtype=float)
game_with_saddle = np.array([[3, 1], [4, 2]], dtype=float)


def analyze_zero_sum_matrix(payoff: np.ndarray) -> dict | None:
    """Calculate the maximin, minimax, and pure saddle points.

    Parameters
    ----------
    payoff:
        The payoff matrix of MAX; MIN receives the negated matrix.
    """

    # Return a dictionary containing five keys as specified below.
    # {"row_minima", "maximin", "column_maxima", "minimax", "saddles"}
    return {
        "row_minima": np.min(payoff, axis=1),
        "maximin": np.max(np.min(payoff, axis=1)),
        "column_maxima": np.max(payoff, axis=0),
        "minimax": np.min(np.max(payoff, axis=0)),
        "saddles": [
            (i, j)
            for i in range(payoff.shape[0])
            for j in range(payoff.shape[1])
            if payoff[i, j] == np.min(payoff[i, :])
            and payoff[i, j] == np.max(payoff[:, j])
        ],
    }


analysis_1 = analyze_zero_sum_matrix(game_without_saddle)
analysis_2 = analyze_zero_sum_matrix(game_with_saddle)

print(f"Game 1: {analysis_1}")
print(f"Game 2: {analysis_2}")

Game 1: {'row_minima': array([-1., -2.]), 'maximin': np.float64(-1.0), 'column_maxima': array([2., 1., 3.]), 'minimax': np.float64(1.0), 'saddles': []}
Game 2: {'row_minima': array([1., 2.]), 'maximin': np.float64(2.0), 'column_maxima': array([4., 2.]), 'minimax': np.float64(2.0), 'saddles': [(1, 1)]}


### Saddle-point reflection

The mismatch between maximin and minimax in Game 1 implies that there is no stable outcome when players are restricted to pure strategies. However, it does not mean the game has no Nash equilibrium; rather, it guarantees that any Nash equilibrium must involve mixed (randomized) strategies.

## Minimax and Alpha–Beta Pruning

> **Problem description.** The root is MAX. Each child is a MIN node, and every number is a terminal utility for MAX. Search children from left to right.

```text
Root MAX
├── MIN: [3, 12, 8]
├── MIN: [2, 4, 6]
└── MIN: [14, 5, 2]
```

Complete both functions so that each returns `(value, visited_leaves)`. Alpha–Beta must return the same value as Minimax while visiting no more leaves.


In [21]:
search_tree = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]


def minimax_tree(node, maximizing: bool) -> tuple[float | None, int | None]:
    """Calculate the Minimax value and count the number of visited leaf nodes.

    Parameters
    ----------
    node:
        A number at a leaf node or a list of child nodes.
    maximizing:
        ``True`` if the current node belongs to MAX, otherwise MIN.
    """

    # Handle leaf nodes, then recurse with alternating MAX/MIN roles.
    if isinstance(node, (int, float)):
        return float(node), 1

    value = float("-inf") if maximizing else float("inf")
    visited_leaves = 0

    for child in node:
        child_value, child_visited_leaves = minimax_tree(child, not maximizing)
        visited_leaves += child_visited_leaves
        if maximizing:
            value = max(value, child_value)
        else:
            value = min(value, child_value)

    return value, visited_leaves


def alphabeta_tree(
    node,
    alpha: float = float("-inf"),
    beta: float = float("inf"),
    maximizing: bool = True,
) -> tuple[float | None, int | None]:
    """Calculate Minimax using Alpha-Beta pruning and count visited leaf nodes.

    Parameters
    ----------
    node:
        A number at a leaf node or a list of child nodes.
    alpha:
        The best lower bound MAX has guaranteed so far.
    beta:
        The best upper bound MIN has guaranteed so far.
    maximizing:
        ``True`` at MAX nodes and ``False`` at MIN nodes.
    """

    # Update alpha/beta and stop when alpha >= beta.
    if isinstance(node, (int, float)):
        return float(node), 1

    value = float("-inf") if maximizing else float("inf")
    visited_leaves = 0

    for child in node:
        child_value, child_visited_leaves = alphabeta_tree(
            child, alpha, beta, not maximizing
        )
        visited_leaves += child_visited_leaves
        if maximizing:
            value = max(value, child_value)
            alpha = max(alpha, value)
        else:
            value = min(value, child_value)
            beta = min(beta, value)

        if alpha >= beta:
            break

    return value, visited_leaves

minimax_value_answer, minimax_visits_answer = minimax_tree(search_tree, True)
alphabeta_value_answer, alphabeta_visits_answer = alphabeta_tree(search_tree)

print(f"Minimax: {minimax_value_answer}, {minimax_visits_answer}")
print(f"Alpha–Beta: {alphabeta_value_answer}, {alphabeta_visits_answer}")

Minimax: 3.0, 9
Alpha–Beta: 3.0, 7


### Pruning reflection

The first branch that can be pruned is the third leaf node (value 6) under the second MIN node. At that moment, the first MIN node has returned a value of 3, setting the root MAX's $\alpha$ to 3. When exploring the second MIN node, the first leaf is 2 and the second leaf is 4, which updates this node's $\beta$ to 2. Since $\alpha = 3 \ge \beta = 2$, pruning becomes valid immediately before visiting the third leaf (6).

# 3. Nash Equilibrium

A strategy profile is a Nash equilibrium when no player can improve their payoff through a unilateral change while all other strategies remain fixed.

In a payoff matrix:

- Fix a column and find the row player's best response.
- Fix a row and find the column player's best response.
- A cell marked as a best response for both players is a pure Nash equilibrium.


## Best responses in the Prisoner's Dilemma

> **Problem description.** Rows belong to Player A and columns belong to Player B. The order is `Cooperate`, `Defect`. Implement reusable functions for best-response masks and pure Nash equilibria.


In [12]:
pd_strategies = ["Cooperate", "Defect"]
pd_payoff_a = np.array([[3, 0], [5, 1]], dtype=float)
pd_payoff_b = np.array([[3, 5], [0, 1]], dtype=float)


def best_response_masks(
    payoff_row: np.ndarray,
    payoff_column: np.ndarray,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Mark best response of row player and column player.

    Parameters
    ----------
    payoff_row, payoff_column:
        Two payoff matrices with same shape.
    """

    # Row player chooses the row (axis 0) that maximizes payoff for each column
    row_max = np.max(payoff_row, axis=0, keepdims=True)
    row_mask = payoff_row == row_max

    # Column player chooses the column (axis 1) that maximizes payoff for each row
    col_max = np.max(payoff_column, axis=1, keepdims=True)
    col_mask = payoff_column == col_max

    return row_mask, col_mask


def pure_nash_equilibria(
    payoff_row: np.ndarray,
    payoff_column: np.ndarray,
) -> list[tuple[int, int]] | None:
    """List coordinations của all pure-strategy Nash equilibrium.

    Parameters
    ----------
    payoff_row, payoff_column:
        Two payoff matrices with same shape.
    """

    row_mask, col_mask = best_response_masks(payoff_row, payoff_column)
    combined = row_mask & col_mask
    indices = np.argwhere(combined)
    return [tuple(idx) for idx in indices]


pd_a_best, pd_b_best = best_response_masks(pd_payoff_a, pd_payoff_b)
pd_equilibria = pure_nash_equilibria(pd_payoff_a, pd_payoff_b)

print(f"Player A best-response mask:\n{pd_a_best}")
print(f"Player B best-response mask:\n{pd_b_best}")
print(f"Pure equilibria: {pd_equilibria}")

Player A best-response mask:
[[False False]
 [ True  True]]
Player B best-response mask:
[[False  True]
 [False  True]]
Pure equilibria: [(np.int64(1), np.int64(1))]


## Dominant strategies and Pareto dominance

> **Problem description.** Use the same Prisoner's Dilemma. A strategy is strictly dominant if it gives a higher payoff than every alternative for every action of the opponent. Profile $x$ Pareto-dominates profile $y$ if neither player is worse off at $x$ and at least one is strictly better off.


In [13]:
def strictly_dominates_strategy(
    payoff: np.ndarray,
    candidate: int,
    alternative: int,
    player: str,
) -> bool | None:
    """Check if a strategy strictly dominates other strategy.

    Parameters
    ----------
    payoff:
        The payoff matrix of the analyzed player.
    candidate, alternative:
        Two strategy indexes to be compared.
    player:
        ``"row"`` or ``"column"``.
    """

    if player == "row":
        return np.all(payoff[candidate, :] > payoff[alternative, :])
    elif player == "column":
        return np.all(payoff[:, candidate] > payoff[:, alternative])
    else:
        raise ValueError


def pareto_dominates(
    payoff_a: np.ndarray,
    payoff_b: np.ndarray,
    profile_x: tuple[int, int],
    profile_y: tuple[int, int],
) -> bool | None:
    """Check profile x Pareto-dominates profile y.

    Parameters
    ----------
    payoff_a, payoff_b:
        Two payoff matrices.
    profile_x, profile_y:
       The two ``(row, column)`` coordinates.
    """

    if not (0 <= profile_x[0] < payoff_a.shape[0] and 0 <= profile_x[1] < payoff_a.shape[1]):
        return False
    if not (0 <= profile_y[0] < payoff_b.shape[0] and 0 <= profile_y[1] < payoff_b.shape[1]):
        return False

    # Payoffs at profiles x and y for both players
    pay_a_x, pay_b_x = payoff_a[profile_x], payoff_b[profile_x]
    pay_a_y, pay_b_y = payoff_a[profile_y], payoff_b[profile_y]

    # x Pareto-dominates y if:
    # 1. No player is worse off: pay_x >= pay_y for both players
    # 2. At least one player is strictly better off: pay_x > pay_y for at least one player
    no_one_worse = (pay_a_x >= pay_a_y) and (pay_b_x >= pay_b_y)
    someone_better = (pay_a_x > pay_a_y) or (pay_b_x > pay_b_y)

    return bool(no_one_worse and someone_better)


a_defect_dominates = strictly_dominates_strategy(pd_payoff_a, 1, 0, "row")
b_defect_dominates = strictly_dominates_strategy(pd_payoff_b, 1, 0, "column")
cooperation_pareto_dominates_defection = pareto_dominates(
    pd_payoff_a,
    pd_payoff_b,
    (0, 0),
    (1, 1),
)

print(f"Defect strictly dominates for A: {a_defect_dominates}")
print(f"Defect strictly dominates for B: {b_defect_dominates}")
print(f"Cooperation Pareto-dominates Defect: {cooperation_pareto_dominates_defection}")

Defect strictly dominates for A: True
Defect strictly dominates for B: True
Cooperation Pareto-dominates Defect: True


### Stability versus social quality

`(Defect, Defect)` is a Nash equilibrium because neither player can unilaterally change their strategy to improve their individual payoff (deviating to Cooperation reduces individual payoff from 1 to 0). On the other hand, `(Cooperate, Cooperate)` is Pareto optimal (socially superior) but unstable because each player has a private, unilateral incentive to defect to get a higher individual payoff (5 instead of 3), assuming the other continues to cooperate.

## Multiple Nash equilibria

> **Problem description.** Two agents prefer to coordinate. The profile `(Up, Left)` gives both agents payoff 2; `(Down, Right)` gives both payoff 1; mismatched actions give both payoff 0.

Reuse `pure_nash_equilibria()` and determine whether either player has a strictly dominant strategy.


In [22]:
coordination_a = np.array([[2, 0], [0, 1]], dtype=float)
coordination_b = np.array([[2, 0], [0, 1]], dtype=float)

# Reuse the helper completed in Exercise 9.
coordination_equilibria = pure_nash_equilibria(coordination_a, coordination_b)

# Assign True or False after comparing both strategies of each player.
a_has_strictly_dominant_strategy = bool(
    strictly_dominates_strategy(coordination_a, 0, 1, "row")
    or strictly_dominates_strategy(coordination_a, 1, 0, "row")
)
b_has_strictly_dominant_strategy = bool(
    strictly_dominates_strategy(coordination_b, 0, 1, "column")
    or strictly_dominates_strategy(coordination_b, 1, 0, "column")
)

print(f"Coordination equilibria: {coordination_equilibria}")
print(f"A has a strictly dominant strategy: {a_has_strictly_dominant_strategy}")
print(f"B has a strictly dominant strategy: {b_has_strictly_dominant_strategy}")

Coordination equilibria: [(np.int64(0), np.int64(0)), (np.int64(1), np.int64(1))]
A has a strictly dominant strategy: False
B has a strictly dominant strategy: False


### Equilibrium-selection reflection

Both equilibria are stable, one effective coordination mechanism that could help the agents select `(Up, Left)` is pre-play communication (cheap talk), where players can announce or agree on their intentions beforehand.

Alternatively, focal points (such as social conventions, cultural norms, or natural saliency) can guide players to coordinate on the Pareto-dominant equilibrium `(Up, Left)`.

## A mixed equilibrium by indifference

> **Problem description.** Consider the asymmetric coordination game below.

$$
A=\begin{bmatrix}2&0\\0&1\end{bmatrix},\qquad
B=\begin{bmatrix}1&0\\0&2\end{bmatrix}.
$$

Let $p$ be the probability that A chooses the first row and $q$ the probability that B chooses the first column. At an interior mixed equilibrium, each player makes the other indifferent between their two pure strategies.

Calculate $p$, $q$, and both players' expected equilibrium utilities.


In [23]:
mixed_game_a = np.array([[2, 0], [0, 1]], dtype=float)
mixed_game_b = np.array([[1, 0], [0, 2]], dtype=float)

# Make B indifferent to find p; make A indifferent to find q.
# B's payoffs: B[0, 0] = 1, B[0, 1] = 0, B[1, 0] = 0, B[1, 1] = 2
# p * 1 + (1 - p) * 0 = p * 0 + (1 - p) * 2 => p = 2/3
p_first_row = 2 / 3

# A's payoffs: A[0, 0] = 2, A[0, 1] = 0, A[1, 0] = 0, A[1, 1] = 1
# q * 2 + (1 - q) * 0 = q * 0 + (1 - q) * 1 => 2q = 1 - q => q = 1/3
q_first_column = 1 / 3

# Use expected_payoff to calculate the equilibrium utility for both players.
strat_row = np.array([p_first_row, 1 - p_first_row])
strat_col = np.array([q_first_column, 1 - q_first_column])

mixed_utility_a = expected_payoff(mixed_game_a, strat_row, strat_col)
mixed_utility_b = expected_payoff(mixed_game_b, strat_row, strat_col)

print(f"p(first row): {p_first_row}")
print(f"q(first column): {q_first_column}")
print(f"A mixed-equilibrium utility: {mixed_utility_a}")
print(f"B mixed-equilibrium utility: {mixed_utility_b}")

p(first row): 0.6666666666666666
q(first column): 0.3333333333333333
A mixed-equilibrium utility: 0.6666666666666667
B mixed-equilibrium utility: 0.6666666666666667


### Indifference reflection

If one pure strategy gave A a strictly higher expected payoff than the other, A would unilaterally maximize their utility by shifting all probability to that better strategy (i.e., playing a pure strategy). Therefore, for A to play a truly mixed strategy with positive probabilities on both options, the expected payoffs of both pure strategies must be exactly equal, making A indifferent.

# 4. Integrated Challenge

## A cybersecurity inspection game

> **Problem description.** A defender chooses `Monitor` or `Ignore`; an attacker simultaneously chooses `Attack` or `Abstain`. Each cell is `(defender payoff, attacker payoff)`.

|  | Attack | Abstain |
|---|---:|---:|
| **Monitor** | $(3,-3)$ | $(-1,0)$ |
| **Ignore** | $(-5,5)$ | $(1,0)$ |

Complete the full analysis:

1. Classify the payoff structure.
2. Find every pure Nash equilibrium.
3. Let $p$ be the probability of `Monitor` and $q$ the probability of `Attack`. Find the mixed equilibrium by indifference.
4. Calculate the expected payoff of each player at the mixed equilibrium.


In [24]:
security_defender = np.array([[3, -1], [-5, 1]], dtype=float)
security_attacker = np.array([[-3, 0], [5, 0]], dtype=float)

# Reuse the helper functions from previous parts.
security_game_type = classify_payoff_structure(security_defender, security_attacker)
security_pure_equilibria = pure_nash_equilibria(security_defender, security_attacker)

# Solve the two indifference equations.
p_monitor = 5 / 8
q_attack = 1 / 5

# Create the two probability vectors and calculate the expected payoffs.
vect1, vect2 = np.array([p_monitor, 1 - p_monitor]), np.array([q_attack, 1 - q_attack])
security_defender_value = expected_payoff(security_defender, vect1, vect2)
security_attacker_value = expected_payoff(security_attacker, vect1, vect2)

print("security payoff structure:", security_game_type)
print("security pure equilibria:", security_pure_equilibria)
print("P(Monitor):", p_monitor)
print("P(Attack):", q_attack)
print("defender equilibrium value:", security_defender_value)
print("attacker equilibrium value:", security_attacker_value)

security payoff structure: general-sum
security pure equilibria: []
P(Monitor): 0.625
P(Attack): 0.2
defender equilibrium value: -0.2
attacker equilibrium value: -5.551115123125783e-17


### Payoff-structure reflection

A competitive situation is not automatically zero-sum because players' interests might not be diametrically opposed, allowing for joint losses or gains. In Game 3 (the Prisoner's Dilemma), if both players defect, they both end up with a payoff of 1 (sum = 2), whereas if they both cooperate, they earn 3 each (sum = 6). Since the sum of payoffs varies depending on the outcome, it is a general-sum game.

### Strategic interpretation

1. The game is not zero-sum because the sum of payoffs is not constant across all cells. For example, `(Monitor, Abstain)` sums to -1, while `(Ignore, Attack)` sums to 0.
2. No pure profile is stable because one player always has an incentive to unilaterally deviate. If the defender monitors, the attacker wants to abstain; if the attacker abstains, the defender wants to ignore; if the defender ignores, the attacker wants to attack; and if the attacker attacks, the defender wants to monitor.
3. The randomized strategies $p$ and $q$ ensure unpredictable behavior. If the defender's policy is predictable (e.g., pure `Ignore`), the attacker can easily exploit this by always attacking. Conversely, if the attacker is predictable, the defender can adjust to minimize waste and perfectly defend.



### Final synthesis

Representation, Minimax, and Nash equilibrium are fundamental concepts that address different aspects of an AI game-playing problem. Representation defines the formal structure of the game, specifying states, actions, transitions, and rules. Minimax provides an optimal decision-making strategy for deterministic, sequential, zero-sum games by assuming a perfectly rational, opposing player. Nash equilibrium expands this analysis to general-sum and simultaneous-move environments, finding stable strategy profiles where no player has a unilateral incentive to deviate. Together, they model, solve, and analyze diverse multi-agent interactions.